# Phase 5B — Time Series Crime Forecasting (Prophet)

**Model:** Facebook Prophet — Bayesian structural time series with Fourier-basis seasonality

**Data:** 60 monthly observations (Jan 2020 – Dec 2024)

**Forecast horizon:** Jan – Dec 2025 (12 months)

**Configuration:**
- Yearly seasonality: Fourier order 10
- Seasonality mode: multiplicative (handles trend × seasonal interaction)
- Changepoint prior scale: 0.15 (moderate flexibility)
- 90% credible interval
- US federal holidays included

**Scope:** Overall crime volume + top-5 categories + top-6 divisions

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Image, display
import subprocess, sys

ROOT   = Path('..').resolve()
PROC   = ROOT / 'data' / 'processed'
REPDIR = ROOT / 'outputs' / 'reports'
FIGDIR = ROOT / 'outputs' / 'figures'

BG = '#0f1117'
plt.rcParams.update({'figure.facecolor': BG, 'axes.facecolor': '#1a1d27',
                     'text.color': 'white', 'axes.labelcolor': 'white'})
print('Setup complete.')

## 0. Run the forecast pipeline

In [ ]:
result = subprocess.run(
    [sys.executable, str(ROOT / 'src' / 'ml_forecast.py')],
    capture_output=True, text=True, cwd=str(ROOT)
)
# Print only the summary section
lines = result.stdout.split('\n')
for line in lines:
    if not line.startswith('17:') and not 'cmdstanpy' in line:
        print(line)

## 1. Overall 12-Month Forecast

In [ ]:
display(Image(filename=str(FIGDIR / 'p5b_01_forecast_overall.png'), width=900))

## 2. Trend & Seasonality Decomposition

In [ ]:
display(Image(filename=str(FIGDIR / 'p5b_02_forecast_components.png'), width=900))

**Component interpretation:**
- **Trend:** Downward slope post-2022 — crime declining after COVID rebound peak
- **Yearly seasonality:** Summer peak (June-August) and secondary December peak; January trough
- **Holidays:** Positive effect around major holidays (more outdoor gatherings = more crime)

## 3. Category-Level Forecasts

In [ ]:
display(Image(filename=str(FIGDIR / 'p5b_03_forecast_categories.png'), width=900))

## 4. Division-Level Forecasts

In [ ]:
display(Image(filename=str(FIGDIR / 'p5b_04_forecast_divisions.png'), width=1000))

## 5. Model Diagnostics

In [ ]:
display(Image(filename=str(FIGDIR / 'p5b_05_forecast_cv.png'), width=900))

## 6. 2025 Predictions Table

In [ ]:
fc = pd.read_csv(REPDIR / 'forecast_2025.csv')
fc['forecast'] = fc['forecast'].astype(int)
fc['ci_lower'] = fc['ci_lower'].astype(int)
fc['ci_upper'] = fc['ci_upper'].astype(int)
fc['range']    = fc['ci_upper'] - fc['ci_lower']

print('2025 Monthly Crime Forecast (Prophet, 90% CI)')
print('=' * 52)
for _, r in fc.iterrows():
    bar_len = int((r['forecast'] - fc['forecast'].min()) /
                   (fc['forecast'].max() - fc['forecast'].min()) * 20)
    bar = '█' * bar_len
    print(f"  {r['month']}  {r['forecast']:,} [{r['ci_lower']:,} - {r['ci_upper']:,}]  {bar}")

print('=' * 52)
print(f"  Annual total (point estimate): {fc['forecast'].sum():,}")
print(f"  Range: {fc['ci_lower'].sum():,} – {fc['ci_upper'].sum():,}")

## 7. Model Performance Summary

| Metric | Value | Interpretation |
|--------|-------|----------------|
| MAE | ~654 crimes/month | Average absolute error on training fit |
| RMSE | ~875 crimes/month | Root mean squared error |
| MAPE | ~5% | Mean absolute percentage error — excellent for public safety forecasting |
| Model | Prophet (Bayesian) | Handles seasonality + COVID structural break automatically |

**Limitations:**
- 60 months of training data is adequate but short for capturing multi-year cycles
- COVID period (2020) acts as an outlier — Prophet handles it via changepoints but the 2025 CI is wider as a result
- External regressors (unemployment, temperature) could improve MAPE further (future work)

**ML implications for Phase 5C:**
- The trend feature (year + month) from the forecast model confirms temporal features are the strongest predictors
- Summer months should receive higher weight in the classifier's training data via sample weighting